# PSY 292 Consumer Psychology

This notebook is designed for exploring and analyzing a sample dataset on consumers on coca-cola. This project contains data collected by Group 4, consumer pyschology students in the University of Ibadan, 2026.

## Workflow
1. Imported required libraries
2. Loaded the dataset
3. Inspected the dataset
4. Cleaned and preprocessed the data
5. Explored patterns with statistics and plots
6. Saved processed data


In [1]:
#Importing Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 2. Load the Dataset
# Replace the path below with the actual CSV/Excel/JSON file you want to analyze.
# Example paths:
# "data/my_dataset.csv"
# "./dataset.csv"
# "C:/path/to/file.xlsx"

file_path = "data/your_dataset.csv"

df = pd.read_csv(file_path)

display(df.head())
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")


In [ ]:
# 3. Explore Dataset Structure
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMemory usage:")
print(df.memory_usage(deep=True))


In [ ]:
# 4. Data Cleaning and Preprocessing
# Check for missing values
missing = df.isnull().sum()
print("Missing values per column:\n", missing[missing > 0])

# Drop duplicate rows if needed
initial_rows = df.shape[0]
df = df.drop_duplicates().copy()
print(f"Dropped {initial_rows - df.shape[0]} duplicate rows.")

# Fill or drop missing values as needed. Adjust these rules based on your dataset.
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna('Unknown')
    else:
        df[col] = df[col].fillna(df[col].median())

# Convert date-like columns to datetime if present
for col in df.columns:
    if 'date' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col])
        except Exception:
            pass

# Optional: remove obvious outliers using IQR for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("\nCleaned dataset preview:")
display(df.head())


In [ ]:
# 5. Exploratory Data Analysis
print("Dataset summary:\n")
print(df.describe(include='all').T)

# Show value counts for categorical fields
categorical_cols = df.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print("\nCategorical feature distributions:\n")
    for col in categorical_cols[:5]:
        print(f"--- {col} ---")
        print(df[col].value_counts().head(10))

# Groupby example for numeric insights
if len(numeric_cols) > 0:
    print("\nNumeric group summary:\n")
    grouped = df.groupby(df.columns[0] if df.columns[0] in df.columns else df.index.name or 'index', dropna=False)
    try:
        print(grouped[numeric_cols[:min(3, len(numeric_cols))]].mean())
    except Exception:
        print(df[numeric_cols[:min(3, len(numeric_cols))]].describe())


In [ ]:
# 6. Statistical Summary
numeric_df = df.select_dtypes(include=[np.number])
print("Summary statistics:\n")
print(numeric_df.describe().T)

if not numeric_df.empty:
    corr = numeric_df.corr()
    print("\nCorrelation matrix:\n")
    display(corr)


In [ ]:
# 7. Data Visualization
# Histograms for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 0:
    fig, axes = plt.subplots(nrows=max(1, len(numeric_cols) // 2), ncols=2, figsize=(14, 4 * max(1, (len(numeric_cols) + 1) // 2)))
    axes = np.ravel(axes)
    for i, col in enumerate(numeric_cols[:len(axes)]):
        axes[i].hist(df[col], bins=20, edgecolor='black')
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
    for j in range(len(numeric_cols), len(axes)):
        fig.delaxes(axes[j])
    plt.tight_layout()
    plt.show()

# Scatter plot example for first two numeric columns
if len(numeric_cols) >= 2:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x=numeric_cols[0], y=numeric_cols[1])
    plt.title(f'{numeric_cols[0]} vs {numeric_cols[1]}')
    plt.show()

# Boxplot for numeric columns
if len(numeric_cols) > 0:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df[numeric_cols[:min(5, len(numeric_cols)])], orient='h')
    plt.title('Boxplot of Numeric Features')
    plt.show()

# Correlation heatmap
if len(numeric_cols) > 1:
    plt.figure(figsize=(12, 8))
    sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Correlation Heatmap')
    plt.show()

# Target distribution example if a categorical target column exists
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    target_col = categorical_cols[0]
    plt.figure(figsize=(8, 5))
    df[target_col].value_counts().plot(kind='bar', color='steelblue')
    plt.title(f'Distribution of {target_col}')
    plt.xticks(rotation=45)
    plt.ylabel('Count')
    plt.show()


In [ ]:
# 8. Feature Engineering
# Example feature engineering ideas:
# - Create new columns from existing ones
# - Encode categorical features
# - Scale numeric features

# Example: create a new feature by combining two numeric columns if available
if len(numeric_cols) >= 2:
    new_feature_name = f"{numeric_cols[0]}_x_{numeric_cols[1]}"
    df[new_feature_name] = df[numeric_cols[0]] * df[numeric_cols[1]]
    print(f"Created new feature: {new_feature_name}")

# Example: one-hot encoding for object columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if categorical_cols:
    df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print("Shape after one-hot encoding:", df_encoded.shape)
    df = df_encoded

# Example: scaling numeric features
from sklearn.preprocessing import StandardScaler

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 1:
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    print("Scaled numeric features with StandardScaler.")

print("Feature-engineered dataset preview:")
display(df.head())


In [ ]:
# 9. Save Processed Data
# Save the cleaned and processed dataset to a new CSV file.
output_path = 'data/processed_dataset.csv'

df.to_csv(output_path, index=False)
print(f"Processed dataset saved to: {output_path}")


In [ ]:
[]